# 10 -- Melon then wheat: switch crops by day, not by price

`melon_rush_agent` (all units grow MELON, no wheat ever) beat
`wheat_loop_melon_agent` 20/20 in `notebooks/09_melon_rush.ipynb` --
but it goes as low as $68 in the early game (no income until day 10)
and keeps replanting MELON forever, even once the market is crashed.

This increment: plant MELON for the first ~10 days, then switch new
plantings to WHEAT from day 10 onward (WHEAT's market barely reacts to
oversupply -- `above_target=0.20`/`log`, vs. MELON's `3.60`/`sq` --
so it never loses value the way MELON does after a big dump). A day
cutoff, not a market-price threshold: melon's `first_yield_day` is
exactly 10, so day 10 is precisely when the first wave becomes
sellable and further melon plantings would only mature very late in
the season anyway.

Three pieces:
1. **What to plant on an empty tile**: `MELON` while `obs["day"] < 10`,
   `WHEAT` from day 10 onward -- same rule for the farmer and every hand.
2. **What seed to buy**: only the crop from (1) -- never buy MELON seed
   once day 10 hits, so nothing goes to waste unused.
3. **How to tend what's already planted**: read the harvest threshold
   from the tile's *own* `crop` field (`tile["crop"]`), not from the
   current day-based phase -- a MELON planted on day 8 still needs to
   reach its own day 18 before harvesting, even though new plantings
   have already switched to WHEAT by then.

In [4]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import (
    DIRECTIONS,
    HAND_ZONES,
    LOOP_ORDER_12,
    LOOP_ORDER_HANDS,
    SPAWN_FEEDER,
    melon_rush_agent,
)

## Building `melon_then_wheat_agent`

Your turn. Start from `melon_rush_agent`'s body (same structure: farmer
on `LOOP_ORDER_12`, hands on `LOOP_ORDER_HANDS`/`HAND_ZONES`, everyone
using the same crop/threshold logic -- just no longer hardcoded to
`"MELON"`/`10`).

For each unit (farmer and every hand):
- Compute `crop_to_plant = "MELON" if obs["day"] < 10 else "WHEAT"` and
  use it wherever the old code planted `"MELON"`.
- Read seed stock and buy seed for `crop_to_plant` only (two separate
  `obs["private"]["seeds"]` lookups -- `WHEAT` and `MELON` -- but only
  one `BUY_SEED` order per turn, for whichever one is current).
- When a tile already has a plant on it, the harvest threshold comes
  from *that tile's* `tile["crop"]` (`10` for `MELON`, `2` for `WHEAT`),
  not from `crop_to_plant` -- those can differ once the day-10 switch
  has happened but some old MELON plants are still growing.
- Sell whichever of `WHEAT`/`MELON` is sitting in the shed (both may be
  present during the transition).

In [ ]:
def melon_then_wheat_agent(obs):
    crop_to_plant = "MELON" if obs["day"] < 10 else "WHEAT"
    crop_seeds = obs["private"]["seeds"].get(crop_to_plant, 0)

    market = []
    hands = []

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **HAND_ZONES[i % len(HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]
        in_own_zone = (hx, hy) in HAND_ZONES[i % len(HAND_ZONES)]
        if hand_tile == "LOCKED" or not in_own_zone:
            op = move
        elif hand_tile is None and crop_seeds > 0:
            op = ["PLANT", crop_to_plant]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            threshold = 10 if hand_tile["crop"] == "MELON" else 2
            if crop_age >= threshold:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = move
        else:
            op = move
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or crop_seeds <= 5:
        market.append(["BUY_SEED", crop_to_plant, 6 - crop_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    melon_in_shed = obs["private"]["shed"].get("MELON", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])
    if melon_in_shed > 0:
        market.append(["SELL", "MELON", 1])

    if tile is None and crop_seeds > 0:
        farmer = ["PLANT", crop_to_plant]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        threshold = 10 if tile["crop"] == "MELON" else 2
        if crop_age >= threshold:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": hands, "market": market}

## Validate against the current best

Current best is `melon_rush_agent`. Run twice, per this project's rule.

In [ ]:
print(evaluate(melon_rush_agent, melon_then_wheat_agent, n=20))

In [ ]:
print(evaluate(melon_rush_agent, melon_then_wheat_agent, n=20))

## Sell all at once, or 1/turn?

`melon_then_wheat_agent` sells `SELL <crop> 1` per turn (inherited from
`wheat_loop_agent`'s original convention). Checked directly in the
installed engine (`kaggriculture.py:560`, the per-unit lockstep loop): a
single `SELL MELON 6` order is processed **one unit at a time**
internally, re-quoting the price after each unit -- so from our own
selling alone, a big batch and 1/turn give identical revenue.

But the market is shared, and episode 96612755 showed the opponent also
sells a lot of melon. Every turn we hold shed inventory instead of
selling it is a turn where *their* selling can degrade the price before
ours goes through. Hypothesis: dumping the whole shed the moment it's
harvested should out-race the opponent's own price erosion, vs. trickling
1/turn. Test it directly.

In [ ]:
def melon_then_wheat_dump_agent(obs):
    crop_to_plant = "MELON" if obs["day"] < 10 else "WHEAT"
    crop_seeds = obs["private"]["seeds"].get(crop_to_plant, 0)

    market = []
    hands = []

    for i, (hx, hy) in enumerate(obs["farms"][obs["player"]]["hands"]):
        hand_loop = {**LOOP_ORDER_HANDS, **SPAWN_FEEDER, **HAND_ZONES[i % len(HAND_ZONES)]}
        target_x, target_y = hand_loop[(hx, hy)]
        if (target_x, target_y) == (hx, hy):
            move = ["PASS"]
        else:
            move = [DIRECTIONS[(target_x - hx, target_y - hy)]]

        hand_tile = obs["farms"][obs["player"]]["tiles"][hy][hx]
        in_own_zone = (hx, hy) in HAND_ZONES[i % len(HAND_ZONES)]
        if hand_tile == "LOCKED" or not in_own_zone:
            op = move
        elif hand_tile is None and crop_seeds > 0:
            op = ["PLANT", crop_to_plant]
        elif hand_tile is not None and hand_tile.get("kind") == "PLANT":
            crop_age = obs["day"] - hand_tile["planted_day"]
            threshold = 10 if hand_tile["crop"] == "MELON" else 2
            if crop_age >= threshold:
                op = ["HARVEST"]
            elif not hand_tile.get("watered_today"):
                op = ["WATER"]
            else:
                op = move
        else:
            op = move
        hands.append(op)

    if len(obs["farms"][obs["player"]]["hands"]) < 5:
        market.append(["HIRE"])

    if obs["step"] == 0 or crop_seeds <= 5:
        market.append(["BUY_SEED", crop_to_plant, 6 - crop_seeds])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    melon_in_shed = obs["private"]["shed"].get("MELON", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", wheat_in_shed])
    if melon_in_shed > 0:
        market.append(["SELL", "MELON", melon_in_shed])

    if tile is None and crop_seeds > 0:
        farmer = ["PLANT", crop_to_plant]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        threshold = 10 if tile["crop"] == "MELON" else 2
        if crop_age >= threshold:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": hands, "market": market}

## Validate: dump vs. 1/turn

Current best is `melon_then_wheat_agent` (1/turn). Run twice.

In [ ]:
print(evaluate(melon_then_wheat_agent, melon_then_wheat_dump_agent, n=20))

In [ ]:
print(evaluate(melon_then_wheat_agent, melon_then_wheat_dump_agent, n=20))